# CES 스케줄 페이지 크롤링

CES 공식 웹사이트의 스케줄 정보를 크롤링합니다.


In [ ]:
# 필요한 라이브러리 설치
# !pip install requests beautifulsoup4 selenium pandas openpyxl


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import json
from datetime import datetime
import time
import re
import urllib3

# SSL 경고 무시 설정 (인증서 검증 문제 해결)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


## 방법 1: requests + BeautifulSoup (정적 페이지)

먼저 기본적인 requests 방법으로 시도합니다.


In [ ]:
# CES 스케줄 페이지 URL
url = "https://www.ces.tech/schedule/"

# 헤더 설정 (브라우저처럼 보이게)
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7',
}

try:
    # SSL 검증 비활성화 (인증서 문제 해결)
    # verify=False는 개발/테스트 환경에서만 사용하세요. 프로덕션 환경에서는 인증서를 올바르게 설정하는 것이 좋습니다.
    response = requests.get(url, headers=headers, timeout=10, verify=False)
    response.raise_for_status()
    print(f"상태 코드: {response.status_code}")
    print(f"페이지 크기: {len(response.text)} bytes")
except Exception as e:
    print(f"에러 발생: {e}")


In [ ]:
# HTML 파싱
soup = BeautifulSoup(response.text, 'html.parser')

# 페이지 구조 확인
print("페이지 제목:", soup.title.string if soup.title else "없음")

# 스케줄 관련 요소 찾기 시도
schedule_items = soup.find_all(['article', 'div'], class_=re.compile(r'schedule|session|event', re.I))
print(f"\n찾은 스케줄 아이템 수: {len(schedule_items)}")

# 시간대별 세션 찾기
time_sections = soup.find_all(['section', 'div'], class_=re.compile(r'time|session', re.I))
print(f"시간대 섹션 수: {len(time_sections)}")

# 샘플 출력 (처음 3개)
if schedule_items:
    for i, item in enumerate(schedule_items[:3]):
        print(f"\n--- 아이템 {i+1} ---")
        print(item.get_text(strip=True)[:200])


## 방법 2: Selenium 사용 (동적 콘텐츠)

페이지가 JavaScript로 동적으로 로드되는 경우 Selenium을 사용합니다.


In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException

# Chrome 옵션 설정
chrome_options = Options()
chrome_options.add_argument('--headless')  # 브라우저 창 숨기기
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')

# 웹드라이버 초기화 (ChromeDriver 필요)
# driver = webdriver.Chrome(options=chrome_options)
# print("Selenium 드라이버 초기화 완료")


In [ ]:
def scrape_ces_schedule_selenium(url, wait_time=10):
    """
    Selenium을 사용하여 CES 스케줄 페이지 크롤링
    """
    driver = None
    try:
        driver = webdriver.Chrome(options=chrome_options)
        driver.get(url)
        
        # 페이지 로딩 대기
        WebDriverWait(driver, wait_time).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )
        
        # 추가 대기 (동적 콘텐츠 로딩)
        time.sleep(3)
        
        # 페이지 소스 가져오기
        html = driver.page_source
        soup = BeautifulSoup(html, 'html.parser')
        
        return soup
        
    except Exception as e:
        print(f"Selenium 에러: {e}")
        return None
    finally:
        if driver:
            driver.quit()

# 사용 예시 (주석 해제하여 실행)
# selenium_soup = scrape_ces_schedule_selenium(url)
# if selenium_soup:
#     print("Selenium으로 페이지 로드 완료")


## 데이터 추출 및 저장

페이지에서 스케줄 정보를 추출하여 데이터프레임과 CSV/Excel로 저장합니다.


In [ ]:
def extract_schedule_data(soup):
    """
    BeautifulSoup 객체에서 스케줄 데이터 추출
    """
    schedules = []
    
    # 날짜별 섹션 찾기
    date_sections = soup.find_all(['section', 'div'], class_=re.compile(r'date|day', re.I))
    
    # 시간대별 세션 찾기
    # 다양한 가능한 선택자 시도
    session_selectors = [
        {'tag': 'div', 'class': re.compile(r'session|event|schedule', re.I)},
        {'tag': 'article'},
        {'tag': 'li', 'class': re.compile(r'session|event', re.I)},
    ]
    
    for selector in session_selectors:
        sessions = soup.find_all(selector['tag'], class_=selector.get('class'))
        if sessions:
            print(f"{selector['tag']} 태그로 {len(sessions)}개 세션 발견")
            break
    
    # 시간 패턴 찾기 (예: "9:00 AM", "10:00 AM-10:40 AM")
    time_pattern = re.compile(r'(\d{1,2}):(\d{2})\s*(AM|PM)', re.I)
    
    # 세션 정보 추출 시도
    all_text_elements = soup.find_all(text=time_pattern)
    
    for element in all_text_elements[:50]:  # 처음 50개만 확인
        parent = element.parent if hasattr(element, 'parent') else None
        if parent:
            # 주변 텍스트 수집
            session_text = parent.get_text(separator=' | ', strip=True)
            
            # 시간 추출
            times = time_pattern.findall(session_text)
            
            if times:
                schedule_data = {
                    'time': ' '.join([f"{t[0]}:{t[1]} {t[2]}" for t in times[:2]]),
                    'description': session_text[:500],
                    'html': str(parent)[:300]
                }
                schedules.append(schedule_data)
    
    return schedules

# 데이터 추출
schedules = extract_schedule_data(soup)
print(f"\n추출된 스케줄 수: {len(schedules)}")

if schedules:
    # 첫 번째 스케줄 샘플 출력
    print("\n=== 첫 번째 스케줄 샘플 ===")
    for key, value in schedules[0].items():
        print(f"{key}: {value[:200]}")


In [ ]:
# 데이터프레임 생성
if schedules:
    df = pd.DataFrame(schedules)
    
    # 데이터 정리
    df['extracted_time'] = df['time'].str.extract(r'(\d{1,2}):(\d{2})\s*(AM|PM)', expand=False)
    
    print(f"\n데이터프레임 크기: {df.shape}")
    print("\n데이터프레임 미리보기:")
    print(df.head())
    
    # CSV로 저장
    output_csv = 'ces_schedule.csv'
    df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"\nCSV 파일 저장 완료: {output_csv}")
    
    # Excel로 저장
    try:
        output_excel = 'ces_schedule.xlsx'
        df.to_excel(output_excel, index=False, engine='openpyxl')
        print(f"Excel 파일 저장 완료: {output_excel}")
    except Exception as e:
        print(f"Excel 저장 실패: {e}")
else:
    print("추출된 스케줄이 없습니다. 페이지 구조를 다시 확인해주세요.")


## 페이지 구조 분석

페이지의 실제 HTML 구조를 확인하여 정확한 선택자를 찾습니다.


In [ ]:
# 페이지 구조 분석을 위한 HTML 일부 저장
html_sample = response.text[:5000]  # 처음 5000자만
print("=== HTML 샘플 (처음 5000자) ===")
print(html_sample)

# 주요 클래스명/ID 찾기
all_classes = set()
all_ids = set()

for tag in soup.find_all(True):
    if tag.get('class'):
        all_classes.update(tag.get('class'))
    if tag.get('id'):
        all_ids.add(tag.get('id'))

# 스케줄 관련 클래스/ID 필터링
schedule_classes = [c for c in all_classes if re.search(r'schedule|session|event|time|date', c, re.I)]
schedule_ids = [i for i in all_ids if re.search(r'schedule|session|event|time|date', i, re.I)]

print("\n=== 스케줄 관련 클래스 ===")
for cls in sorted(schedule_classes)[:20]:
    print(f"  .{cls}")

print("\n=== 스케줄 관련 ID ===")
for id_val in sorted(schedule_ids)[:20]:
    print(f"  #{id_val}")


## 고급: API 엔드포인트 확인

최신 웹사이트는 종종 API를 통해 데이터를 제공합니다. 네트워크 요청을 확인하여 API 엔드포인트를 찾을 수 있습니다.


In [ ]:
# 페이지에 포함된 JavaScript 코드에서 API 엔드포인트 찾기
script_tags = soup.find_all('script')
api_endpoints = []

for script in script_tags:
    if script.string:
        # API URL 패턴 찾기
        api_patterns = [
            r'https?://[^"\s]+/api/[^"\s]+',
            r'https?://[^"\s]+/schedule[^"\s]+',
            r'fetch\(["\']([^"\']+)["\']',
            r'axios\.(get|post)\(["\']([^"\']+)["\']',
        ]
        
        for pattern in api_patterns:
            matches = re.findall(pattern, script.string)
            if matches:
                api_endpoints.extend(matches if isinstance(matches[0], str) else [m[1] if len(m) > 1 else m[0] for m in matches])

if api_endpoints:
    print("발견된 API 엔드포인트:")
    for endpoint in set(api_endpoints)[:10]:
        print(f"  {endpoint}")
else:
    print("API 엔드포인트를 찾지 못했습니다.")

# JSON 데이터 직접 찾기
json_data = soup.find_all('script', type='application/json')
if json_data:
    print(f"\nJSON 데이터 스크립트 태그 발견: {len(json_data)}개")
    for i, json_script in enumerate(json_data[:3]):
        try:
            data = json.loads(json_script.string)
            print(f"\nJSON 데이터 {i+1}:")
            print(json.dumps(data, indent=2, ensure_ascii=False)[:500])
        except:
            print(f"JSON 파싱 실패: {i+1}")


## 개선된 데이터 추출 함수

실제 페이지 구조를 분석한 후 정확한 선택자로 데이터를 추출합니다.


In [ ]:
def extract_ces_schedule_improved(soup):
    """
    CES 스케줄 페이지에서 구조화된 데이터 추출
    """
    all_schedules = []
    
    # 날짜별 그룹 찾기 (예: "Mon, Jan 05")
    date_headers = soup.find_all(['h2', 'h3', 'div'], string=re.compile(r'Mon|Tue|Wed|Thu|Fri|Sat|Sun', re.I))
    
    # 시간대별 세션 찾기 (예: "9:00 AM", "10:00 AM")
    time_elements = soup.find_all(string=re.compile(r'\d{1,2}:\d{2}\s*(AM|PM)', re.I))
    
    print(f"날짜 헤더 발견: {len(date_headers)}개")
    print(f"시간 요소 발견: {len(time_elements)}개")
    
    # 각 시간 요소의 부모 요소에서 세션 정보 추출
    for time_elem in time_elements[:100]:  # 처음 100개만 처리
        try:
            # 부모 요소 찾기
            parent = time_elem.parent
            if not parent:
                continue
            
            # 상위 컨테이너 찾기 (세션 정보가 있는 영역)
            session_container = parent.find_parent(['article', 'div', 'li', 'section'])
            if not session_container:
                session_container = parent
            
            # 세션 제목 찾기
            title_elem = session_container.find(['h3', 'h4', 'h5', 'a'], class_=re.compile(r'title|name|heading', re.I))
            if not title_elem:
                title_elem = session_container.find(['h3', 'h4', 'h5'])
            
            # 장소 찾기
            location_elem = session_container.find(string=re.compile(r'ARIA|LVCC|Venetian|Sphere|Fontainebleau', re.I))
            
            # 설명 찾기
            desc_elem = session_container.find(['p', 'div'], class_=re.compile(r'description|summary|abstract', re.I))
            
            # 토픽/트랙 찾기
            topics = []
            topic_elems = session_container.find_all(['span', 'div', 'a'], class_=re.compile(r'topic|tag|category|track', re.I))
            topics = [t.get_text(strip=True) for t in topic_elems]
            
            session_data = {
                'time': time_elem.strip() if time_elem else '',
                'title': title_elem.get_text(strip=True) if title_elem else '',
                'location': location_elem.strip() if location_elem else '',
                'description': desc_elem.get_text(strip=True)[:500] if desc_elem else '',
                'topics': ', '.join(topics[:5]) if topics else '',
                'full_text': session_container.get_text(separator=' | ', strip=True)[:1000]
            }
            
            # 제목이 있는 경우만 추가
            if session_data['title'] or session_data['full_text']:
                all_schedules.append(session_data)
                
        except Exception as e:
            continue
    
    return all_schedules

# 개선된 추출 함수 실행
improved_schedules = extract_ces_schedule_improved(soup)
print(f"\n개선된 방법으로 추출된 스케줄 수: {len(improved_schedules)}")

if improved_schedules:
    print("\n=== 첫 번째 스케줄 (개선된 방법) ===")
    for key, value in improved_schedules[0].items():
        print(f"{key}: {str(value)[:200]}")


## 최종 결과 저장

개선된 방법으로 추출한 데이터를 CSV와 Excel로 저장합니다.


In [ ]:
# 최종 데이터프레임 생성 및 저장
if improved_schedules:
    final_df = pd.DataFrame(improved_schedules)
    
    # 중복 제거 (같은 제목과 시간)
    final_df = final_df.drop_duplicates(subset=['title', 'time'], keep='first')
    
    # 시간 정렬
    def parse_time(time_str):
        """시간 문자열을 파싱하여 정렬 가능한 형식으로 변환"""
        if not time_str:
            return (0, 0)
        match = re.search(r'(\d{1,2}):(\d{2})\s*(AM|PM)', time_str, re.I)
        if match:
            hour = int(match.group(1))
            minute = int(match.group(2))
            am_pm = match.group(3).upper()
            if am_pm == 'PM' and hour != 12:
                hour += 12
            elif am_pm == 'AM' and hour == 12:
                hour = 0
            return (hour, minute)
        return (0, 0)
    
    final_df['time_sort'] = final_df['time'].apply(parse_time)
    final_df = final_df.sort_values('time_sort')
    final_df = final_df.drop('time_sort', axis=1)
    
    print(f"\n최종 데이터프레임 크기: {final_df.shape}")
    print("\n최종 데이터프레임 미리보기:")
    print(final_df.head(10).to_string())
    
    # CSV로 저장
    output_csv = 'ces_schedule_final.csv'
    final_df.to_csv(output_csv, index=False, encoding='utf-8-sig')
    print(f"\n✅ CSV 파일 저장 완료: {output_csv}")
    
    # Excel로 저장
    try:
        output_excel = 'ces_schedule_final.xlsx'
        final_df.to_excel(output_excel, index=False, engine='openpyxl')
        print(f"✅ Excel 파일 저장 완료: {output_excel}")
        
        # 컬럼 너비 자동 조정 (선택사항)
        from openpyxl import load_workbook
        wb = load_workbook(output_excel)
        ws = wb.active
        for column in ws.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            adjusted_width = min(max_length + 2, 50)
            ws.column_dimensions[column_letter].width = adjusted_width
        wb.save(output_excel)
        print("✅ Excel 컬럼 너비 자동 조정 완료")
        
    except Exception as e:
        print(f"❌ Excel 저장 실패: {e}")
    
    # 통계 정보 출력
    print(f"\n=== 통계 정보 ===")
    print(f"총 세션 수: {len(final_df)}")
    if not final_df['location'].empty:
        print(f"\n장소별 세션 수:")
        print(final_df['location'].value_counts().head(10))
    if not final_df['topics'].empty:
        print(f"\n주요 토픽:")
        all_topics = []
        for topics_str in final_df['topics'].dropna():
            if topics_str:
                all_topics.extend([t.strip() for t in topics_str.split(',')])
        from collections import Counter
        topic_counts = Counter(all_topics)
        for topic, count in topic_counts.most_common(10):
            print(f"  {topic}: {count}개")
            
else:
    print("⚠️ 추출된 스케줄이 없습니다.")
    print("페이지가 동적으로 로드되는 경우 Selenium을 사용해야 할 수 있습니다.")
    print("위의 Selenium 코드를 주석 해제하여 실행해보세요.")
